In [ ]:
import logging
import os
import cv2
import numpy as np
from sklearn.cluster import KMeans
import json
import psycopg2

# Configuration log
logging.basicConfig(level=logging.INFO)

# Setting the root directory for image storage
image_root_dir = os.getenv("IMAGE_ROOT_DIR", "/Users/Tommy/AI/image-search/clothing-images/")

# image_root_dir = os.getenv("IMAGE_ROOT_DIR",
#                            "/Users/Tommy/MSC-TUD/Team_Project/dataset/image/zalando-hd-resized/test/cloth")

# database configuration
db_config = {
    'dbname': os.getenv('DB_NAME', 'image-search'),
    'user': os.getenv('DB_USER', 'postgres'),
    'password': os.getenv('DB_PASSWORD', 'test-postgres'),
    'host': os.getenv('DB_HOST', '127.0.0.1'),
}

# batch size for inserting into the database
batch_size = int(os.getenv("BATCH_SIZE", 100))

# database connection
conn = psycopg2.connect(**db_config)
cur = conn.cursor()

# Batch data storage
batch_data = []

# Iterate through the image catalog
for subdir, _, files in os.walk(image_root_dir):
    for filename in files:
        if filename.endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(subdir, filename)
            image_name = filename

            try:
                # Read image
                image_np = cv2.imread(image_path)
                if image_np is None:
                    logging.warning(f"Failed to load image: {image_path}")
                    continue

                # Convert image to HSV
                hsv_image = cv2.cvtColor(image_np, cv2.COLOR_BGR2HSV)
                pixels = hsv_image.reshape((-1, 3))  # Flatten image data

                # Apply KMeans to find main colors
                n_colors = 3  # Default to 3 colors
                kmeans = KMeans(n_clusters=n_colors, random_state=42)
                kmeans.fit(pixels)

                # Get the main colors and their proportions
                main_colors = kmeans.cluster_centers_
                color_proportions = np.bincount(kmeans.labels_) / len(kmeans.labels_)

                # Sort the colors by proportion (largest to smallest)
                sorted_indices = np.argsort(-color_proportions)
                main_colors = main_colors[sorted_indices]
                color_proportions = color_proportions[sorted_indices]

                # Convert the main colors and proportions to JSON format
                main_colors_json = json.dumps(main_colors.tolist())
                color_proportions_json = json.dumps(color_proportions.tolist())

                # Add to Batch
                batch_data.append((image_name, main_colors_json, color_proportions_json))

                # If batch size is reached, insert into the database
                if len(batch_data) >= batch_size:
                    cur.executemany("""
                        INSERT INTO image_main_colors (image_name, main_colors, color_proportions)
                        VALUES (%s, %s, %s)
                    """, batch_data)
                    conn.commit()
                    logging.info(f"Inserted {len(batch_data)} records into database.")
                    batch_data = []  # Clear the batch for the next set of data

            except Exception as e:
                logging.error(f"Error processing image {image_name}: {e}")

# Insert remaining data
if batch_data:
    cur.executemany("""
        INSERT INTO image_main_colors (image_name, main_colors, color_proportions)
        VALUES (%s, %s, %s)
    """, batch_data)
    conn.commit()
    logging.info(f"Inserted remaining {len(batch_data)} records into database.")

# Close the database connection
cur.close()
conn.close()
logging.info("All images processed and database connection closed.")


INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records into database.
INFO:root:Inserted 100 records int